In [ ]:
import argparse
import re
import pandas as pd
from pathlib import Path
from collections import defaultdict
import json
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import matplotlib.dates as mdates
from datetime import datetime


In [ ]:
input_folder = Path("/Users/kix/Work/ibram/05_csv_stats/")
output_folder = Path("/Users/kix/Work/ibram/05_csv_stats/output2/")

In [ ]:
output_files = {
    "ip_proto-pdf": (input_folder / "protocol_counts.json", output_folder / "protocol_counts.pdf"),
    "ip_proto-tex": (input_folder / "protocol_counts.json", output_folder / "protocol_counts.tex"),
    "ip_src-pdf": (input_folder / "ip_src_counts.json", output_folder / "ip_src_counts.pdf"),
    "ip_dst-pdf": (input_folder / "ip_dst_counts.json", output_folder / "ip_dst_counts.pdf"),
    "ip_dst_counts-pdf": (input_folder / "ip_dst_counts.json", output_folder / "ip_dst_counts_counts.pdf"),
    "ip_dst-tex": (input_folder / "ip_dst_counts.json", output_folder / "ip_dst_counts.tex"),
    "ip_5min-pdf": (input_folder / "ip_5min_counts.json", output_folder / "ip_5min_counts.pdf"),
    "udp_dports-tex": (input_folder / "udp_dport_counts.json", output_folder / "udp_dport_counts.tex"),
    "tcp_dports-tex": (input_folder / "tcp_dport_counts.json", output_folder / "tcp_dport_counts.tex"),
    "tcp_flags-tex": (input_folder / "tcp_flag_counts.json", output_folder / "tcp_flag_counts.tex"),
    "icmp_types-tex": (input_folder / "icmp_type_counts.json", output_folder / "icmp_type_counts.tex"),
    "icmp_types_and_codes-tex": (input_folder / "icmp_types_and_codes_counts.json", output_folder / "icmp_types_and_codes_counts.tex"),
    "ip_top25_ip_src-tex": (input_folder / "ip_src_counts.json", output_folder / "ip_top25_ip_src.tex"),
}

In [ ]:
PROTO_NAMES = {
    1: "  1 - ICMP",
    4: "  4 - IP",
    6: "  6 - TCP",
    8: "  8 - EGP",
    17: " 17 - UDP",
    41: " 41 - IPv6",
    47: " 47 - GRE",
    50: " 50 - ESP",
    103: "103 - PIM",
    132: "132 - SCTP",
    138: "138 - MPLS",
}

PALETTE = ["#5c94bd", "#ff7f0e", "#49e4dcc0", "#dd5a5a", "#e7eb30", "#8c564b", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]

In [ ]:
# Create output directory if it doesn't exist
output_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
def plot_protocols_stacked(json_file, output_file, max_protocols=10, days_step=10):
    fontsize = 20
    with open(json_file, "r") as f:
        data = json.load(f)

    # Convert protocol keys to int
    new_data = {day: {int(proto): count for proto, count in proto_counts.items()}
                for day, proto_counts in data.items()}

    df = pd.DataFrame(new_data).T
    if df.empty:
        print("No data to plot.")
        return

    df.index.name = "day"
    df.fillna(0, inplace=True)

    # Sort columns by protocol number
    df = df.reindex(sorted(df.columns), axis=1)

    # Rename columns
    df.rename(columns=lambda x: PROTO_NAMES.get(x, str(x)), inplace=True)

    # Get the max_protocols most common protocols across all days
    protocol_totals = df.sum(axis=0)
    top_protocols = protocol_totals.nlargest(max_protocols).index
    df = df[top_protocols]

    # Date format DD-MM-YYYY
    dt_index = pd.to_datetime(df.index, format="%Y%m%d")
    df.index = dt_index.strftime("%d-%m-%Y")

    # Plot
    ax = df.plot(
        kind="bar",
        stacked=True,
        figsize=(14, 8),
        color=PALETTE[:len(df.columns)],
        width=0.85,
    )

    # Show only every days_step days (optional). Remove this block if you want all days.
    tick_positions = list(range(len(df.index)))
    visible_positions = tick_positions[::days_step]
    ax.set_xticks(visible_positions)
    ax.set_xticklabels([df.index[i] for i in visible_positions], rotation=45, ha="right")

    # Axes and formatting
    ax.set_xlabel("Day", fontsize=fontsize + 2)
    ax.set_ylabel("Packets (millions)", fontsize=fontsize + 2)
    ax.tick_params(axis='x', labelsize=fontsize + 2)
    ax.tick_params(axis='y', labelsize=fontsize + 2)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: (y / 1_000_000)))

    # Legend outside at the top
    ax.legend(title="Protocol",
               fontsize=fontsize,
               title_fontsize=fontsize,
               loc='lower center',
               labelspacing=0.2,
               handletextpad=0.2,
               borderpad=0.1,
               bbox_to_anchor=(0.5, 0.98),
               ncols=5)

    plt.subplots_adjust(top=0.30)  # space for legend
    plt.tight_layout()

    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
plot_protocols_stacked(output_files["ip_proto-pdf"][0], output_files["ip_proto-pdf"][1])


In [ ]:
# This function read the 5 minutes intervals and draw a line plot of packets per 5 minutes interval, all days in one plot
# using only onle line for all days
def plot_5min_traffic(json_file, output_file, ymax=None, log_scale=False):
    fontsize = 20
    # Load data
    with open(json_file, "r") as f:
        data = json.load(f)

    # Create DataFrame
    all_data = []
    for _, time_counts in data.items():
        for time_str, count in time_counts.items():
            # time_str already contains full date and time (YYYY-MM-DD HH:MM:SS)
            timestamp = datetime.strptime(time_str, "%Y-%m-%d %H:%M:%S")
            all_data.append((timestamp, count))

    df = pd.DataFrame(all_data, columns=["timestamp", "count"])
    df = df[["timestamp", "count"]]
    df.set_index("timestamp", inplace=True)
    df.sort_index(inplace=True)

    print(df.head())

    # Draw line chart
    plt.figure(figsize=(14,9))
    ax = plt.gca()
    ax.plot(df.index, df['count'], marker='o', linestyle='-')

    # Remove x margins
    ax.margins(x=0)

    if log_scale:
        ax.set_yscale("log")

    # Y Max
    if ymax is None:
        y_max = df['count'].max() * 1.1  # Add 10% headroom
    else:
        y_max = ymax

    ax.set_ylim(0, y_max)

    # Axes and formatting
    ax.set_xlabel("Day", fontsize=fontsize + 2)
    ax.set_ylabel("Packets (thousands)", fontsize=fontsize + 2)
    ax.tick_params(axis='x', labelsize=fontsize + 2)
    ax.tick_params(axis='y', labelsize=fontsize + 2)

    if log_scale:
        ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{int(y):,}"))
        # plt.title("Packets per 5-Minute Interval (Log Scale)", fontsize=fontsize + 4)
    else:
        ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{int(y / 1_000):,}"))
        # plt.title("Packets per 5-Minute Interval", fontsize=fontsize + 4)

    plt.xticks(rotation=45)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Plot 5-minute traffic using the correct JSON (ip_5min_counts.json)
plot_5min_traffic(output_files["ip_5min-pdf"][0], output_files["ip_5min-pdf"][1], ymax=None, log_scale=True)

In [ ]:
# Create a latex table with the top 25 TCP ports, with rank, port, count, and percentage of total
def create_tcp_ports_latex_table(json_file, output_file, top_values=25, rank_column=False):
    # Load data
    with open(json_file, "r") as f:
        data = json.load(f)

    # Aggregate counts across all days
    total_counts = defaultdict(int)
    for day, port_counts in data.items():
        for port, count in port_counts.items():
            total_counts[int(port)] += int(count)

    # Sort ports by count (descending) and take top N
    sorted_ports = sorted(total_counts.items(), key=lambda x: x[1], reverse=True)[:top_values]
    total_packets = sum(total_counts.values())

    # Create LaTeX table
    with open(output_file, "w") as f:
        if rank_column:
            f.write("\\begin{tabular}{|r|r|r|c|}\n")
            f.write("\\hline\n")
            f.write("\\hline\n")
            f.write("Rank & TCP Port & Count & Percentage \\\\\n")
            f.write("\\hline\n")
        else:
            f.write("\\begin{tabular}{|r|r|c|}\n")
            f.write("\\hline\n")
            f.write("\\hline\n")
            f.write("TCP Port & Count & Percentage \\\\\n")
            f.write("\\hline\n")

        sum_percent = 0
        sum_packets = 0

        for rank, (port, count) in enumerate(sorted_ports, start=1):
            percentage = (count / total_packets) * 100
            sum_percent += percentage
            sum_packets += count
            if rank_column:
                f.write(f"{rank:3} & {port:>5} & {count:>10,} & {percentage:6.2f}\\% \\\\\n")
            else:
                f.write(f"{port:>5} & {count:>10,} & {percentage:6.2f}\\% \\\\\n")

        f.write("\\hline\n")
        # Use multicol to span the first 3 columns for the total row
        if rank_column:
            f.write(f"Total & {sum_packets:>10,} & {sum_percent:>6.2f}\\% \\\\\n")
        else:
            f.write(f"Total & {sum_packets:>10,} & {sum_percent:>6.2f}\\% \\\\\n")
        f.write("\\hline\n")
        f.write("\\end{tabular}\n")

    print(f"LaTeX table saved to {output_file}")

In [ ]:
create_tcp_ports_latex_table(output_files["tcp_dports-tex"][0], output_files["tcp_dports-tex"][1])

In [ ]:
# Create a latex table with the top 25 UDP ports, with rank, port, count, and percentage of total
def create_udp_ports_latex_table(json_file, output_file, top_values=25, rank_column=False):
    # Load data
    with open(json_file, "r") as f:
        data = json.load(f)

    # Aggregate counts across all days
    total_counts = defaultdict(int)
    for day, port_counts in data.items():
        for port, count in port_counts.items():
            total_counts[int(port)] += int(count)

    # Sort ports by count (descending) and take top N
    sorted_ports = sorted(total_counts.items(), key=lambda x: x[1], reverse=True)[:top_values]
    total_packets = sum(total_counts.values())

    # Create LaTeX table
    with open(output_file, "w") as f:
        if rank_column:
            f.write("\\begin{tabular}{|r|r|r|c|}\n")
            f.write("\\hline\n")
            f.write("\\hline\n")
            f.write("Rank & UDP Port & Count & Percentage \\\\\n")
            f.write("\\hline\n")
        else:
            f.write("\\begin{tabular}{|r|r|c|}\n")
            f.write("\\hline\n")
            f.write("\\hline\n")
            f.write("UDP Port & Count & Percentage \\\\\n")
            f.write("\\hline\n")

        sum_percent = 0
        sum_packets = 0

        for rank, (port, count) in enumerate(sorted_ports, start=1):
            percentage = (count / total_packets) * 100
            sum_percent += percentage
            sum_packets += count
            if rank_column:
                f.write(f"{rank:3} & {port:>5} & {count:>10,} & {percentage:6.2f}\\% \\\\\n")
            else:
                f.write(f"{port:>5} & {count:>10,} & {percentage:6.2f}\\% \\\\\n")

        f.write("\\hline\n")
        # Use multicol to span the first 3 columns for the total row
        if rank_column:
            f.write(f"Total & {sum_packets:>10,} & {sum_percent:>6.2f}\\% \\\\\n")
        else:
            f.write(f"Total & {sum_packets:>10,} & {sum_percent:>6.2f}\\% \\\\\n")

        f.write("\\hline\n")
        f.write("\\hline\n")
        f.write("\\end{tabular}\n")

    print(f"LaTeX table saved to {output_file}")

In [ ]:
create_udp_ports_latex_table(output_files["udp_dports-tex"][0], output_files["udp_dports-tex"][1])

In [ ]:
# This function creates a LaTeX table for TCP flag combinations
# One column per flag: A (ACK), C (CWR), E (ECE), F (FIN), N (NS), P (PSH), R (RST), S (SYN), U (URG)
# Shows letter when flag is active, space when inactive (ordered alphabetically)
# Plus columns for Rank, Count, and Percentage

def create_tcp_flags_latex_table(json_file, output_file, top_n=25, rank_column=False):
    no_selected_flag_character = '   '
    with open(json_file, "r") as f:
        data = json.load(f)

    # Aggregate counts across all days
    total_counts = defaultdict(int)
    for _, combo_counts in data.items():
        for combo, count in combo_counts.items():
            total_counts[combo] += int(count)

    # Sort combos by count desc and take top N
    sorted_combos = sorted(total_counts.items(), key=lambda x: x[1], reverse=True)[:top_n]
    total_packets = sum(total_counts.values()) if total_counts else 0

    # Parse combination string into flag dict
    def parse_flags(combo_str):
        # Returns dict with True/False for each flag
        flags = {'A': False, 'C': False, 'E': False, 'F': False, 'N': False,
                 'P': False, 'R': False, 'S': False, 'U': False}
        if combo_str == 'NONE':
            return flags

        parts = combo_str.split(',')
        for p in parts:
            if p == 'ACK': flags['A'] = True
            elif p == 'CWR': flags['C'] = True
            elif p == 'ECE': flags['E'] = True
            elif p == 'FIN': flags['F'] = True
            elif p == 'NS': flags['N'] = True
            elif p == 'PSH': flags['P'] = True
            elif p == 'RST': flags['R'] = True
            elif p == 'SYN': flags['S'] = True
            elif p == 'URG': flags['U'] = True
        return flags

    with open(output_file, 'w') as f:
        # Header with one column per flag (alphabetically ordered)
        if rank_column:
            f.write("\\begin{tabular}{|c|c|c|c|c|c|c|c|c|r|c|}\n")
            f.write("\\hline\n")
            f.write("\\hline\n")
            f.write("Rank & A & C & E & F & N & P & R & S & U & Count & \\% \\\\\n")
            f.write("\\hline\n")
        else:
            f.write("\\begin{tabular}{|c|c|c|c|c|c|c|c|c|r|c|}\n")
            f.write("\\hline\n")
            f.write("\\hline\n")
            f.write(" A & C & E & F & N & P & R & S & U & Count & \\% \\\\\n")
            f.write("\\hline\n")

        sum_percent = 0
        sum_packets = 0

        for rank, (combo, count) in enumerate(sorted_combos, start=1):
            percentage = (count / total_packets * 100) if total_packets else 0
            sum_percent += percentage
            sum_packets += count
            flags = parse_flags(combo)

            # Build row: rank, then each flag column (letter or space), count, percentage
            # Flags in alphabetical order: A, C, E, F, N, P, R, S, U
            row = f"{rank:>3}" if rank_column else ""
            for flag_letter in ['A', 'C', 'E', 'F', 'N', 'P', 'R', 'S', 'U']:
                cell = f" {flag_letter} " if flags[flag_letter] else no_selected_flag_character
                row += f"&{cell}" if rank_column else f"{cell}&"
            row += f" {count:>12,} & {percentage:6.2f}\\%"
            f.write(row + " \\\\\n")

        f.write("\\hline\n")
        # Use multicol to span the first 9 columns for the total row
        if rank_column:
            f.write(f"\\multicolumn{{10}}{{|l|}}{{Total}} {'':6} & {sum_packets:>12,} & {sum_percent:>6.2f}\\% \\\\\n")
        else:
            f.write(f"\\multicolumn{{9}}{{|l|}}{{Total}} {'':6} & {sum_packets:>12,} & {sum_percent:>6.2f}\\% \\\\\n")
        f.write("\\hline\n")
        f.write("\\hline\n")
        f.write("\\end{tabular}\n")

    print(f"LaTeX TCP flags table saved to {output_file}")



In [ ]:
# Generate the table
create_tcp_flags_latex_table(output_files["tcp_flags-tex"][0], output_files["tcp_flags-tex"][1])

In [ ]:
# This function creates a LaTeX table for ICMP types and codes
def create_icmp_types_codes_latex_table(json_file, output_file, top_n=25, rank_column=False):
    with open(json_file, "r") as f:
        data = json.load(f)

    # Aggregate counts across all days
    total_counts = defaultdict(int)
    for day, type_code_counts in data.items():
        for type_code_str, count in type_code_counts.items():
            type_code = eval(type_code_str)  # Convert string back to tuple
            total_counts[type_code] += int(count)

    # Sort by count desc and take top N
    sorted_type_codes = sorted(total_counts.items(), key=lambda x: x[1], reverse=True)[:top_n]
    total_packets = sum(total_counts.values()) if total_counts else 0

    with open(output_file, 'w') as f:
        f.write("\\hline\n")
        f.write("\\hline\n")
        if rank_column:
            f.write("\\begin{tabular}{|c|c|c|r|c|}\n")
            f.write("Rank & ICMP Type & ICMP Code & Count & \\% \\\\\n")
        else:
            f.write("\\begin{tabular}{|c|c|r|c|}\n")
            f.write("ICMP Type & ICMP Code & Count & \\% \\\\\n")
        f.write("\\hline\n")

        sum_percent = 0
        sum_packets = 0

        for rank, ((icmp_type, icmp_code), count) in enumerate(sorted_type_codes, start=1):
            percentage = (count / total_packets * 100) if total_packets else 0
            sum_percent += percentage
            sum_packets += count
            if rank_column:
                f.write(f"{rank:>3} & {icmp_type:2} & {icmp_code:2} & {count:>12,} & {percentage:6.2f}\\% \\\\\n")
            else:
                f.write(f"{icmp_type:2} & {icmp_code:2} & {count:>12,} & {percentage:6.2f}\\% \\\\\n")

        f.write("\\hline\n")

        if rank_column:
            # Use multicol to span the first 3 columns for the total row
            f.write(f"\\multicolumn{{3}}{{|l|}}{{Total}} & {sum_packets:>10,} & {sum_percent:>6.2f}\\% \\\\\n")
        else:
            # Use multicol to span the first 2 columns for the total row
            f.write(f"\\multicolumn{{2}}{{|l|}}{{Total}} & {sum_packets:>12,} & {sum_percent:>6.2f}\\% \\\\\n")
        f.write("\\hline\n")
        f.write("\\end{tabular}\n")

    print(f"LaTeX ICMP types and codes table saved to {output_file}")

In [ ]:
create_icmp_types_codes_latex_table(output_files["icmp_types_and_codes-tex"][0], output_files["icmp_types_and_codes-tex"][1])

In [ ]:
# This LaTeX table shows the ip protocol counts aggregated across all days and include the percentage of total packets for each protocol.
def create_ip_proto_latex_table(json_file, output_file):
    with open(json_file, "r") as f:
        data = json.load(f)

    # Aggregate counts across all days
    total_counts = defaultdict(int)
    for day, proto_counts in data.items():
        for proto, count in proto_counts.items():
            total_counts[int(proto)] += int(count)

    # Sort by protocol number
    sorted_protos = sorted(total_counts.items(), key=lambda x: x[0])
    total_packets = sum(total_counts.values()) if total_counts else 0

    with open(output_file, 'w') as f:
        f.write("\\begin{tabular}{|l|r|c|}\n")
        f.write("\\hline\n")
        f.write("\\hline\n")
        f.write("Protocol & Count & \\% \\\\\n")
        f.write("\\hline\n")

        sum_percent = 0
        sum_packets = 0

        for proto, count in sorted_protos:
            percentage = (count / total_packets) * 100 if total_packets else 0
            sum_percent += percentage
            sum_packets += count
            proto_name = PROTO_NAMES.get(proto, str(proto))
            f.write(f"{proto_name:<15} ({proto:>2}) & {count:>12,} & {percentage:6.2f}\\% \\\\\n")

        f.write("\\hline\n")
        f.write(f"Total {'':<15} & {sum_packets:>12,} & {sum_percent:>6.2f}\\% \\\\\n")
        f.write("\\hline\n")
        f.write("\\end{tabular}\n")

    print(f"LaTeX IP protocol table saved to {output_file}")

In [ ]:
create_ip_proto_latex_table(output_files["ip_proto-tex"][0], output_files["ip_proto-tex"][1])

In [ ]:
# This LaTeX table shows the ip protocol counts aggregated across all days and include the percentage of total packets for each protocol.
def create_ip_proto_latex_table_resume(json_file, output_file, min_count=100):
    # Rename output file to include "_resume" before the extension
    output_file = output_file.with_name(output_file.stem + "_resume" + output_file.suffix)

    with open(json_file, "r") as f:
        data = json.load(f)

    # Aggregate counts across all days
    total_counts = defaultdict(int)
    for _, proto_counts in data.items():
        for proto, count in proto_counts.items():
            total_counts[int(proto)] += int(count)

    # Sort by protocol number
    sorted_protos = sorted(total_counts.items(), key=lambda x: x[0])
    total_packets = sum(total_counts.values()) if total_counts else 0

    # Rest counter
    rest_protocol_sum = 0
    rest_protocol_count = 0

    with open(output_file, 'w') as f:
        f.write("\\begin{tabular}{|l|r|c|}\n")
        f.write("\\hline\n")
        f.write("\\hline\n")
        f.write("Protocol & Count & \\% \\\\\n")
        f.write("\\hline\n")

        for proto, count in sorted_protos:
            if count < min_count:
                rest_protocol_sum += count
                rest_protocol_count += 1
                continue
            pct = (count / total_packets * 100) if total_packets else 0
            proto_name = PROTO_NAMES.get(proto, str(proto))
            proto_name = proto_name.split(" - ")[-1]  # Keep only the name part, remove the number
            proto_name = proto_name.strip()  # Remove any leading/trailing whitespace
            proto_name = f"{proto_name} ({proto})"  # Add protocol number in parentheses
            f.write(f"{proto_name:<21} & {count:>12,} & {pct:6.2f}\\% \\\\\n")

        if rest_protocol_sum > 0:
            pct = (rest_protocol_sum / total_packets * 100) if total_packets else 0
            f.write(f"Other {rest_protocol_count} protocols {'':1} & {rest_protocol_sum:>12,} & {pct:6.2f}\\% \\\\\n")
            f.write("\\hline\n")
            f.write(f"Total {'':<15} & {total_packets:>12,} & 100.00\\% \\\\\n")

        f.write("\\hline\n")
        f.write("\\hline\n")
        f.write("\\end{tabular}\n")

    print(f"LaTeX IP protocol table saved to {output_file}")

In [ ]:
create_ip_proto_latex_table_resume(output_files["ip_proto-tex"][0], output_files["ip_proto-tex"][1])

In [ ]:
# This function reads the ip_dst_counts.json and aggregates the counts across all days,
# then creates a plot of the 256 destinations as a bar chart.
def plot_top_ip_dst(json_file, output_file):
    fontsize = 16
    with open(json_file, "r") as f:
        data = json.load(f)

    # Remove the "155.54.96." string from the keys if present
    cleaned_data = {}
    for day, ip_counts in data.items():
        cleaned_ip_counts = {}
        for ip_dst, count in ip_counts.items():
            if ip_dst.startswith("155.54.96."):
                cleaned_ip = ip_dst[len("155.54.96."):]
            else:
                cleaned_ip = ip_dst
            cleaned_ip_counts[cleaned_ip] = count
        cleaned_data[day] = cleaned_ip_counts

    # Aggregate counts across all days
    total_counts = defaultdict(int)
    for day, ip_counts in cleaned_data.items():
        for ip_dst, count in ip_counts.items():
            total_counts[ip_dst] += int(count)

    # Sort the dictionary by the key (is integer form of the last octet)
    total_counts = dict(sorted(total_counts.items(), key=lambda x: int(x[0])))

    print(total_counts)

    # Sort by count desc and take top 256
    # sorted_ips = sorted(total_counts.items(), key=lambda x: x[1], reverse=True)[:256]

    # Create DataFrame for plotting
    df = pd.DataFrame(total_counts.items(), columns=["ip_dst", "count"])
    df.set_index("ip_dst", inplace=True)

    # Print the median and mean of the counts
    print(f"Median of counts: {df['count'].median()}")
    print(f"Mean of counts: {df['count'].mean()}")
    print(f"Minimum of counts: {df['count'].min()}")
    print(f"Maximum of counts: {df['count'].max()}")

    # Plot bar chart
    plt.figure(figsize=(16, 8))
    ax = df.plot(kind="bar", legend=False)

    # Show in the X-axis the xticks every 16 ticks
    plt.xticks(ticks=range(0, len(df.index), 16), labels=df.index[::16], rotation=90)

    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y / 1_000_000:,}"))

    plt.xlabel("Destination IP - Last Octet", fontsize=fontsize)
    plt.ylabel("Packet Count (millions)", fontsize=fontsize)
    # plt.title("Top 256 Destination IPs by Packet Count", fontsize=fontsize)
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    # print(f"Plotted top 256 destination IPs.")

In [ ]:
plot_top_ip_dst(output_files["ip_dst_counts-pdf"][0], output_files["ip_dst_counts-pdf"][1])

In [ ]:
# Top 25 IP source IP addresses as a LaTeX table
def create_top_ip_src_latex_table(json_file, output_file, top_n=25, rank_column=False):
    with open(json_file, "r") as f:
        data = json.load(f)

    # Aggregate counts across all days
    total_counts = defaultdict(int)
    for day, ip_counts in data.items():
        for ip_src, count in ip_counts.items():
            total_counts[ip_src] += int(count)

    # Sort by count desc and take top N
    sorted_ips = sorted(total_counts.items(), key=lambda x: x[1], reverse=True)[:top_n]
    total_packets = sum(total_counts.values()) if total_counts else 0

    with open(output_file, 'w') as f:
        if rank_column:
            f.write("\\begin{tabular}{|c|l|r|c|}\n")
            f.write("\\hline\n")
            f.write("\\hline\n")
            f.write("Rank & Source IP & Count & \\% \\\\\n")
            f.write("\\hline\n")
        else:
            f.write("\\begin{tabular}{|l|r|c|}\n")
            f.write("\\hline\n")
            f.write("Source IP & Count & \\% \\\\\n")
            f.write("\\hline\n")

        sum_percent = 0
        sum_packets = 0

        for rank, (ip_src, count) in enumerate(sorted_ips, start=1):
            percentage = (count / total_packets) * 100 if total_packets else 0
            sum_percent += percentage
            sum_packets += count
            pct = percentage
            if rank_column:
                f.write(f"{rank} & {ip_src:<15} & {count:>12,} & {pct:6.2f}\\% \\\\\n")
            else:
                f.write(f"{ip_src:<15} & {count:>12,} & {pct:6.2f}\\% \\\\\n")

        f.write("\\hline\n")
        f.write(f"Total {'':<9} & {sum_packets:>12,} & {sum_percent:>6.2f}\\% \\\\\n")
        f.write("\\hline\n")
        f.write("\\hline\n")
        f.write("\\end{tabular}\n")

    print(f"LaTeX top source IPs table saved to {output_file}")


In [ ]:
create_top_ip_src_latex_table(output_files["ip_top25_ip_src-tex"][0], output_files["ip_top25_ip_src-tex"][1])